In [1]:
# 6-21-2026

In [55]:
import pandas as pd
import joblib
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [56]:
# adjust hyperparameters by experimenting with a single domain. same hyperparams will be used for each domain (identical model)

In [57]:
domain_id = "22"

# load raw train and test splits for this domain, it's around average for domain dataset size
X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
X_test = pd.read_csv(f"test_X/domain_{domain_id}.csv")
y_test = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [58]:
scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")

In [59]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [60]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=5
) # make eval set to find strong hyperparams

In [61]:
XGB_PARAMS = {
    "n_estimators": 1000,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0,
    "reg_lambda": 2.0,
    "random_state": 5,
    "n_jobs": -1,
    "early_stopping_rounds": 30,
    "eval_metric": "rmse"
}

In [62]:
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,30
,enable_categorical,False
,eval_metric,'rmse'


In [63]:
print(f"best iteration: {model.best_iteration}")

best iteration: 998


In [64]:
train_pred = model.predict(X_train_scaled)
test_pred = model.predict(X_test_scaled)

train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

In [65]:
print(f"train mse: {train_mse:.4f}, test mse: {test_mse:.4f}")
print(f"train r2: {train_r2:.4f}, test r2: {test_r2:.4f}")

train mse: 0.5405, test mse: 0.6348
train r2: 0.2798, test r2: 0.1168


In [ ]:
from scipy.stats import spearmanr

# spearman correlation, robust to outliers and scale, tests if relative ordering is captured
train_spearman, _ = spearmanr(y_train, train_pred)
test_spearman, _ = spearmanr(y_test, test_pred)

print(f"train spearman: {train_spearman:.4f}, test spearman: {test_spearman:.4f}")

train spearman: 0.5378, test spearman: 0.4171


In [66]:
# time to make T for xgb. using hyperparams above. T(i,j) includes BOTH r2 and spearman, will choose best one later

In [67]:
import os
import glob
import pandas as pd
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

In [68]:
domain_files = glob.glob("train_X/domain_*.csv")
domain_ids = sorted(
    int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))
    for f in domain_files
)
print(f"found {len(domain_ids)} domains")

found 34 domains


In [ ]:
models = {}
scalers = {}

for domain_id in domain_ids:
    X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
    y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]

    scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
    X_train_scaled = scaler.transform(X_train)

    # make  a validation split out of train only, for early stopping
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=5
    )

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    models[domain_id] = model
    scalers[domain_id] = scaler

    print(f"domain {domain_id} done, best iteration: {model.best_iteration}")
# takes ~3min

domain 0 done, best iteration: 999
domain 1 done, best iteration: 237
domain 2 done, best iteration: 854
domain 4 done, best iteration: 997
domain 5 done, best iteration: 999
domain 6 done, best iteration: 183
domain 7 done, best iteration: 34
domain 8 done, best iteration: 285
domain 11 done, best iteration: 999
domain 12 done, best iteration: 991
domain 13 done, best iteration: 998
domain 16 done, best iteration: 604
domain 18 done, best iteration: 28
domain 19 done, best iteration: 201
domain 20 done, best iteration: 999
domain 21 done, best iteration: 616
domain 22 done, best iteration: 999
domain 23 done, best iteration: 461
domain 25 done, best iteration: 808
domain 26 done, best iteration: 259
domain 27 done, best iteration: 459
domain 28 done, best iteration: 353
domain 29 done, best iteration: 349
domain 30 done, best iteration: 85
domain 32 done, best iteration: 166
domain 33 done, best iteration: 153
domain 36 done, best iteration: 55
domain 37 done, best iteration: 999
doma

In [70]:
# preload all test sets once, dont rereading csvs 34 times per domain
test_X_raw = {}
test_y = {}

for domain_id in domain_ids:
    test_X_raw[domain_id] = pd.read_csv(f"test_X/domain_{domain_id}.csv")
    test_y[domain_id] = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [71]:
# T(i, j): i is source domain (model trained on i), j is target domain (evaluated on j)
T_r2 = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)
T_spearman = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)

In [72]:
for i in domain_ids:
    model_i = models[i]
    scaler_i = scalers[i]

    for j in domain_ids:
        # apply source domain's scaler to target domain's raw test X, not target's own scaler
        X_test_scaled = scaler_i.transform(test_X_raw[j])
        y_true = test_y[j]

        preds = model_i.predict(X_test_scaled)

        T_r2.loc[i, j] = r2_score(y_true, preds)
        T_spearman.loc[i, j], _ = spearmanr(y_true, preds)

    print(f"finished evaluating source domain {i} against all targets")

finished evaluating source domain 0 against all targets
finished evaluating source domain 1 against all targets
finished evaluating source domain 2 against all targets
finished evaluating source domain 4 against all targets
finished evaluating source domain 5 against all targets
finished evaluating source domain 6 against all targets
finished evaluating source domain 7 against all targets
finished evaluating source domain 8 against all targets
finished evaluating source domain 11 against all targets
finished evaluating source domain 12 against all targets
finished evaluating source domain 13 against all targets
finished evaluating source domain 16 against all targets
finished evaluating source domain 18 against all targets
finished evaluating source domain 19 against all targets
finished evaluating source domain 20 against all targets
finished evaluating source domain 21 against all targets
finished evaluating source domain 22 against all targets
finished evaluating source domain 23 ag

In [73]:
T_r2

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.082427,0.034094,-0.288698,-0.040650,0.010981,-0.024899,-0.029905,-0.202515,-0.234495,-0.080118,...,-0.018745,0.015248,-0.005301,-0.138215,-0.013082,-0.067857,-0.174737,-0.024440,-0.129056,-0.018218
1,-0.049389,0.068142,-1.788817,-0.123606,0.016822,-0.169196,-0.070191,-0.216617,-0.340486,-0.097380,...,-0.081827,-0.254489,-0.198603,-0.209785,-0.052981,-0.331123,-0.369709,-0.606237,-0.276050,-0.100021
2,-0.115285,-0.190382,0.075777,-0.090529,-0.130250,-0.085343,-0.100113,-0.064745,-0.038646,-0.142555,...,-0.027653,-0.084303,-0.182896,-0.103508,-0.153819,-0.232714,-0.122705,-0.123080,-0.071605,-0.149801
4,-0.098142,-0.102119,-0.410604,0.087469,-0.087289,0.008562,-0.154285,-0.584487,-0.411731,-0.121630,...,-0.027462,-0.065558,-0.080825,-0.713473,-0.072826,-0.079169,-0.534115,0.055546,-0.055237,-0.130105
5,-0.339169,-0.230913,-1.258072,-0.191959,0.125674,-0.292566,-0.116537,-0.082744,-0.349741,-0.072399,...,-0.411716,-1.026649,-0.193658,-0.216796,-0.145929,-0.386605,-0.421089,-0.202131,-0.269240,-0.295926
6,-0.068408,-0.071725,-0.652696,-0.003475,-0.106622,0.059485,-0.286598,-0.181577,-0.479734,-0.384989,...,-0.125172,-0.097048,-0.087522,-0.298219,-0.124889,-0.764458,-0.606936,0.069531,-0.148878,-0.026325
7,0.008537,0.001917,-0.181663,-0.000501,-0.000673,-0.005259,0.030110,0.005251,-0.150641,-0.032696,...,-0.005378,0.026027,0.018644,-0.120265,0.012850,-0.015915,-0.135141,-0.000779,-0.043981,-0.003711
8,-0.372454,-0.154512,-2.025085,-0.347098,-0.185794,-0.318065,-0.306370,0.090917,-1.641845,-0.378282,...,-0.338486,-0.964653,-0.682659,-0.916422,-0.502449,-1.515899,-1.258605,-0.630560,-0.573015,-0.187269
11,-0.034508,-0.070830,-0.187940,-0.140611,-0.001103,-0.127517,-0.070401,-0.114925,0.149585,-0.104913,...,0.018515,-0.080093,-0.109668,0.078993,-0.083297,-0.250602,-0.078791,-0.113316,-0.108201,-0.092546
12,-0.326098,-0.089600,-1.661298,-0.353880,-0.172932,-0.223342,-0.312461,-0.961145,-0.640611,0.151868,...,-0.519515,-0.606168,-0.240704,-0.366958,-0.501335,-0.397650,-0.910107,-0.495559,-0.169381,-0.450016


In [74]:
T_spearman

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.340479,0.268365,0.023255,0.073371,0.203848,0.084109,0.144361,0.091938,0.172346,0.041892,...,0.189594,0.219888,0.123013,0.275676,0.093795,0.008626,0.097742,0.101453,0.053211,0.039357
1,0.130374,0.363223,0.004915,0.026436,0.196109,0.028512,0.146295,0.032563,0.148122,0.053528,...,0.211636,0.060401,0.025993,0.235225,0.129532,0.043776,0.108788,0.102869,0.027382,-0.006507
2,0.158666,0.179357,0.299131,0.083269,0.147714,0.072678,0.092537,0.145160,0.159003,0.051983,...,0.176229,0.165970,0.096143,0.211042,0.032863,0.076805,0.042326,0.054624,0.086426,-0.032850
4,0.176460,0.233192,0.070815,0.339380,0.104151,0.182697,0.081252,0.172169,0.068067,0.029840,...,0.152971,0.193996,0.125479,0.242181,0.132990,-0.004366,0.023557,0.296029,0.078820,0.065330
5,0.057727,0.097891,0.026106,0.002197,0.408792,0.040854,0.080465,0.202452,0.166424,0.075737,...,0.127916,0.134689,0.061794,0.244491,0.083776,0.016166,0.113704,0.108696,0.016897,0.035414
6,0.155405,0.206558,-0.032054,0.196678,0.135194,0.261384,0.076180,0.117966,0.091738,0.037575,...,0.150062,0.086082,0.010821,0.227637,0.096021,-0.014234,0.130803,0.289311,0.104805,0.032214
7,0.171326,0.120688,0.114499,0.072293,0.128848,0.062304,0.253052,0.114621,0.199106,0.030878,...,0.150272,0.208196,0.146168,0.240127,0.185270,0.004546,0.066115,0.014426,-0.002759,0.019736
8,0.064002,0.141572,-0.069144,-0.008793,0.033965,0.042129,0.088350,0.375348,-0.107754,0.074146,...,0.063773,0.038667,-0.029178,-0.117891,-0.036877,0.041773,-0.019745,0.174362,0.012393,-0.003695
11,0.203778,0.177833,0.048010,0.043103,0.213477,0.018920,0.084285,0.115636,0.451009,0.034828,...,0.231801,0.131167,0.069483,0.372411,0.180345,-0.022681,0.138525,0.062533,-0.007456,0.046920
12,0.064277,0.124809,0.013145,0.051965,0.088640,0.058783,0.010582,-0.013618,0.145340,0.437233,...,0.107291,-0.078648,0.056119,0.178570,-0.040708,0.051727,0.091536,0.044845,0.095220,-0.023619


In [75]:
# check if T(i,j) variance is driven by domain size rather than real transfer differences
domain_sizes = {d: len(pd.read_csv(f"train_X/domain_{d}.csv")) for d in domain_ids}

row_variance = T_spearman.var(axis=1)  # variance of each source domain's row across all targets
sizes = pd.Series(domain_sizes)

print(pd.concat([sizes.rename("train_size"), row_variance.rename("spearman_row_var")], axis=1).sort_values("train_size"))

    train_size  spearman_row_var
30        1825          0.005085
7         3882          0.004902
33        4878          0.007099
39        5817          0.005660
18        6910          0.006116
19        8102          0.005825
36        8936          0.004677
32        9909          0.007301
6        12588          0.007440
38       13423          0.004752
26       13601          0.006355
46       15258          0.009401
49       15598          0.004943
47       21231          0.008569
29       23199          0.006048
16       23340          0.005352
8        26427          0.008334
28       28653          0.004918
1        32690          0.007577
12       34539          0.007923
23       39176          0.007246
27       46920          0.007002
25       68170          0.009961
21       76967          0.004703
13       84219          0.008385
2        91063          0.004633
22      103595          0.007636
5       128217          0.007223
4       140541          0.006346
20      15

In [ ]:
# does variance across SOURCE models for a given TARGET correlate with the target's test set size
target_sizes = {d: len(pd.read_csv(f"test_X/domain_{d}.csv")) for d in domain_ids}

col_variance = T_spearman.var(axis=0)  # variance down each target column, across all 34 source models
sizes_col = pd.Series(target_sizes)

print(pd.concat([sizes_col.rename("test_size"), col_variance.rename("spearman_col_var")], axis=1).sort_values("test_size"))

    test_size  spearman_col_var
30        609          0.004060
7        1294          0.004015
33       1626          0.008221
39       1940          0.004163
18       2304          0.007409
19       2701          0.004744
36       2979          0.003039
32       3303          0.006397
6        4197          0.004156
38       4475          0.008956
26       4534          0.005400
46       5086          0.009321
49       5200          0.003389
47       7078          0.006678
29       7733          0.005321
16       7781          0.009543
8        8809          0.007056
28       9551          0.002960
1       10897          0.005580
12      11513          0.006283
23      13059          0.005245
27      15641          0.007508
25      22724          0.017677
21      25656          0.005052
13      28074          0.008311
2       30355          0.005882
22      34532          0.005707
5       42740          0.005083
4       46847          0.007243
20      51022          0.006693
37      

In [77]:
print("row variance vs train size:", row_variance.corr(sizes))
print("col variance vs test size:", T_spearman.var(axis=0).corr(pd.Series(target_sizes)))

row variance vs train size: 0.5499224468159704
col variance vs test size: 0.2301080586880731


In [78]:
# spearman seems to be the stronger evaluator of domain performance, so will be going with that

In [79]:
T_spearman.to_csv("transfer_matrix_spearman.csv")